<a href="https://www.kaggle.com/code/ionelaralucacrisan/eda-on-retail-data?scriptVersionId=351071878" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Perform exploratory data analysis on retail data with Python
- Coursera Project


# Introduction:

... for the moment the plan is to define the objectives, intention, - load the data, it is available - then discuss the analytical steps

... data source https://archive.ics.uci.edu/dataset/352/online+retail

Info on the variables:

* Variable Name	- Role	-  Type	  -         Description	
* InvoiceNo	  -  ID	 -     Categorical	-   a 6-digit integral number uniquely assigned to each transaction. 
* StockCode	 -   ID	   -   Categorical	 -  a 5-digit integral number uniquely assigned to each distinct product
* Description	-    Feature	-  Categorical	-   product name		
* Quantity	-    Feature	-  Integer	  -     the quantities of each product (item) per transaction
* InvoiceDate	-    Feature	-  Date	     -      the day and time when each transaction was generated
* UnitPrice	  -  Feature	-  Continuous	-   product price per unit	sterling
* CustomerID	-    Feature	-  Categorical	-   a 5-digit integral number uniquely assigned to each customer	
* Country	  -      Feature	-  Categorical	-   the name of the country where each customer resides


# Retreive the data

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/online-retail-xlsx/Online Retail.xlsx


In [2]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt 

# Get acquainted with the data 

In [23]:
OR = pd.read_excel('/kaggle/input/online-retail-xlsx/Online Retail.xlsx')

OR.head(7)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom


In [24]:
# last 6 rows 

OR.tail(6)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541903,581587,23256,CHILDRENS CUTLERY SPACEBOY,4,2011-12-09 12:50:00,4.15,12680.0,France
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [25]:
print(len(OR))                #total rows 

print((OR['Quantity'] < 0).sum())  #total rows with negative quantity 

print(OR['InvoiceNo'].astype(str).str.startswith('C').sum())    #canceled invoices  

print((OR['UnitPrice'] <= 0).sum())  #rows with zero or negative price (group 2: accounting/junk).

541909
10624
9288
2517


In [26]:
adj = OR[(OR['Quantity'] < 0) & ~OR['InvoiceNo'].astype(str).str.startswith('C')]   #no customer id 
print(len(adj), adj['CustomerID'].isna().sum(), (adj['UnitPrice'] == 0).sum())  #most frequent descriptions 
adj['Description'].value_counts().head(10)  # top 10 negative descriptions without C in front 

1336 1336 1336


Description
check                     120
damages                    45
damaged                    42
?                          41
sold as set on dotcom      20
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
??                          7
damages?                    5
Name: count, dtype: int64

In [27]:
# canceled invoices with prices under or equal to 0 (zero) 

((OR['InvoiceNo'].astype(str).str.startswith('C')) & (OR['UnitPrice'] <= 0)).sum() 

np.int64(0)

In [28]:
is_cancel = OR['InvoiceNo'].astype(str).str.startswith('C')

returns = OR[is_cancel]                              # group 1

junk    = OR[(OR['UnitPrice'] <= 0)]                 # group 2

sales   = OR[~is_cancel & (OR['UnitPrice'] > 0)]     # valid  sales 

print(len(returns), len(junk), len(sales))

9288 2517 530104


......   discutat cu 80000 hours.org pana aici 

In [5]:
#types of data 

OR.dtypes

StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

In [6]:
#check the number of entries 

OR.shape

(541909, 7)

In [7]:
#summary statistics 

OR[['Quantity', 'UnitPrice']].describe()

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


# Clean and Validate the Data 

In [8]:
# check for missing 

OR.isnull().sum()

# we have missing for description and customer ID, what do we do with the ones for description - try to drop the missing for description and keep the ones for ID 

StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [10]:
# drop the missing 

#OR = OR.dropna()    # Dropping the missing values.
OR_clean = OR.dropna(subset=OR.columns.difference(['ClientID']))

OR.count()

OR_clean = OR.dropna(subset=['Description'])


In [11]:
cols_except_id = [col for col in OR.columns if col != 'ClientID']
OR_clean = OR.dropna(subset=cols_except_id)


In [12]:
OR_clean = OR.dropna(subset=OR.columns.difference(['ClientID'])) \
             .dropna(subset=['Description'])


In [13]:
# check for missing 

OR.isnull().sum()

# we have missing for description and customer ID, what do we do with the ones for description 

StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [ ]:
#check for duplicates 

OR.duplicated().sum()


In [14]:
# the duplicates, another way 

duplicate_rows_OR = OR[OR.duplicated()]
print("number of duplicate rows: ", duplicate_rows_OR.shape)


number of duplicate rows:  (5848, 7)


In [15]:
#drop the duplicates

OR = OR.drop_duplicates()
OR.head(5)

,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
InvoiceNo,,,,,,,
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [16]:
OR.duplicated().sum()

np.int64(0)

** CHECK FOR OUTLIERS **

In [18]:

len(OR) 

536061

In [19]:
(OR['Quantity'] < 0).sum()

np.int64(10585)

In [ ]:
# detect range values for columns of the dataset 

OR[['Quantity', 'UnitPrice']].describe([x*0.1 for x in range(4)])

In [ ]:
sns.boxplot(x=OR['Quantity'])

# think about how to deal with returned merchandise

In [ ]:
#histogram

plt.hist(OR[['Quantity']])
plt.show()

In [ ]:
# deal with outliers 

# 

In [ ]:
# unable to calculate correlation - sth to do with data types 

In [ ]:
# list of products 

OR_product = OR['Description']

print(OR_product)

... what are the times with the highest sales?

In [ ]:
# new column with the revenue 

OR['value'] = OR.Quantity * OR.UnitPrice

OR.head()

In [ ]:
# calculate income for each day 

income_each_day = OR.groupby("InvoiceDate")['value'].sum()

print(income_each_day)

In [ ]:
# Data visualisation - revenue for each day 

income_each_day.plot(kind='bar')
plt.title("Revenue for each day")
plt.xlabel("Invoice Date")
plt.ylabel("Revenue")
plt.show()



In [ ]:

# Data visualisation - revenue for one in five day 

income_each_day.plot(kind='bar')
plt.title("Revenue for each day")
plt.xlabel("Invoice Date")
plt.ylabel("Revenue")
plt.show()

# trying to get a more readable graph

ax = income_each_day.plot(kind='bar', figsize=(14, 6))
ax.set_xticks(ax.get_xticks()[::100])  # one in 100 days 
plt.show()


In [ ]:
#first 50 products according to value

top_sales = OR.groupby('Description')['value'].sum()

top_50_sales = top_sales.sort_values(ascending=False).head(50)

top_50_sales.plot(kind='bar', figsize=(12,6))
plt.title(" First 50 Products in the Order of Sales Value")
plt.xlabel("Description")
plt.ylabel("Value")
plt.xticks(rotation=90)
plt.show()

In [ ]:
income_each_day.plot(
    x="Invoice Date",
    y="Revenue for Each Day",
    kind="line",
    marker="o"
)
plt.show()

# the highest sales are at the end of the year, that is normal for gifts

In [ ]:
# calculate income for each product

income_each_object = OR.groupby("Description")['value'].sum()

print(income_each_object)

In [ ]:
# calculate income for each day

income_each_day = OR.groupby("InvoiceDate")['value'].sum()

print(income_each_day)

In [ ]:
# bar chart - de corectat 

top_sales = OR.groupby('Description')['value'].sum()
top_sales.plot(kind='bar')
plt.show()

In [ ]:
description_list = (
            OR["Description"]
            .value_counts()
            .loc[lambda x: x > 1]
            .index
            .tolist()
)

len(description_list)

# we have diffrerent 3698 products  in our database

In [ ]:
customer_country_list = (
            OR["Country"]
            .value_counts()
            .loc[lambda x: x > 1]
            .index
            .tolist()
)

print(customer_country_list)